# **Home Exercise 1 on Text Classification**
Implement a Recurrent Neural Network model (Vanilla RNN, GRU, and LSTM) to predict whether a review is positive or negative.

**Data**: [IMDB Dataset of 50K Movie Reviews](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews) (the last 10% of rows serve as the test set).
Compare the performance of the three models.


In [1]:
# Import libraries
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
import os
import sys
import numpy as np
import pandas as pd

import re
from collections import Counter
import matplotlib.pyplot as plt
from datetime import datetime

print("The last time this project was run is:", datetime.now().strftime("%H:%M:%S %d/%m/%Y"))


if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    print(f"Using device: CUDA - {gpu_name}")
    print(f"CUDA Capability: {torch.cuda.get_device_capability(0)}")
else:
    device = torch.device("cpu")
    print("Using device: CPU (CUDA is not available)")


The last time this project was run is: 17:39:11 20/11/2025
Using device: CUDA - Tesla T4
CUDA Capability: (7, 5)


### Download dataset

In [14]:
import kagglehub
import shutil
import os

# Download dataset
src_root = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
print("Downloaded to:", src_root)

# Tìm file CSV bên trong folder
csv_file = None
for root, dirs, files in os.walk(src_root):
    for f in files:
        if f.lower().endswith(".csv"):
            csv_file = os.path.join(root, f)
            break
    if csv_file:
        break

if not csv_file:
    raise FileNotFoundError("Không tìm thấy file CSV trong dataset KaggleHub!")

print("Found CSV:", csv_file)

dest_dir = "/mnt/d/code_for_fun/Python/NLP/NaturalLanguageProcessing/NaturalLanguageProcessing/recurrent_model/TextClassification/data/"
os.makedirs(dest_dir, exist_ok=True)

# Copy file
shutil.copy(csv_file, dest_dir)

print("Copied to:", dest_dir)


Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Downloaded to: /kaggle/input/imdb-dataset-of-50k-movie-reviews
Found CSV: /kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv
Copied to: /mnt/d/code_for_fun/Python/NLP/NaturalLanguageProcessing/NaturalLanguageProcessing/recurrent_model/TextClassification/data/


In [3]:
# Hyperparameter
MAX_VOCAB_SIZE = 25000
MAX_SEQ_LEN = 200
BATCH_SIZE = 64
EMBEDDING_DIM = 100
HIDDEN_DIM = 256
OUTPUT_DIM = 1
N_LAYERS = 2
DROPOUT = 0.5
LEARNING_RATE = 1e-3
NUM_EPOCHS = 5

## Loading data with DataLoader into DataFrame

In [4]:
class IMDBDataset(Dataset):
    def __init__(self, texts, labels, vocab=None, max_len=MAX_SEQ_LEN, is_train=True):
        self.texts = texts
        self.labels = labels
        self.max_len = max_len
        
        if is_train:
            self.vocab = self.build_vocab(texts)
        else:
            self.vocab = vocab
            
    def build_vocab(self, texts):
        all_words = []
        for text in texts:
            all_words.extend(tokenizer(clean_text(text)))
        
        count = Counter(all_words)
        sorted_words = count.most_common(MAX_VOCAB_SIZE)
        
        vocab = {w: i+2 for i, (w, c) in enumerate(sorted_words)}
        vocab['<PAD>'] = 0
        vocab['<UNK>'] = 1
        return vocab
    
    def text_to_sequence(self, text):
        tokens = tokenizer(clean_text(text))
        seq = [self.vocab.get(token, 1) for token in tokens]

        if len(seq) < self.max_len:
            seq = seq + [0] * (self.max_len - len(seq)) # Padding
        else:
            seq = seq[:self.max_len] # Truncating
        return torch.tensor(seq, dtype=torch.long)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        return self.text_to_sequence(text), torch.tensor(label, dtype=torch.float)

## Preprocessing data

In [9]:
import nltk
from bs4 import BeautifulSoup
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

In [15]:
def preprocessing(text):
    soup = BeautifulSoup(text, 'html.parser')
    text = soup.get_text()
    
    # Remove URL and Email
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\S*@\S*\s?', '', text)
    
    # Remove uncommon character
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    
    # Lowercase
    text = text.lower()
    
    # Split into tokens
    tokens = text.split()
    
    # Remove stopwords and lemmatization
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))
    
    if 'not' in stop_words:
        stop_words.remove('not')
        
    clean_tokens = [
        lemmatizer.lemmatize(token) 
        for token in tokens 
        if token not in stop_words and len(token) > 2 # Bỏ từ quá ngắn
    ]
    
    return clean_tokens